<a href="https://colab.research.google.com/github/TheKerbecs/KISysteme26/blob/main/block_1/Aufgabe_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 - Matrix multiplication in Numba


We consider the problem of evaluating the matrix multiplication $C = A\times B$ for matrices $A, B\in\mathbb{R}^{n\times n}$.
A simple Python implementation of the matrix-matrix product is given below through the function `matrix_product`. At the end this
function is checked against the Numpy implementation of the matrix-matrix product.

In [13]:
import numpy as np

def matrix_product(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in range(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

a = np.random.randn(10, 10)
b = np.random.randn(10, 10)

c_actual = matrix_product(a, b)
c_expected = a @ b

error = np.linalg.norm(c_actual - c_expected) / np.linalg.norm(c_expected)
print(f"The error is {error}.")


The error is 1.2914813047309106e-16.


In [14]:
import numba
print(f"Numba version: {numba.__version__}")

Numba version: 0.60.0


The matrix product is one of the most fundamental operations on modern computers. Most algorithms eventually make use of this operation. A lot of effort is therefore spent on optimising the matrix product. Vendors provide hardware optimised BLAS (Basis Linear Algebra Subroutines) that provide highly efficient versions of the matrix product. Alternatively, open-source libraries sucha as Openblas provide widely used generic open-source implementations of this operation.

In this assignment we want to learn at the example of matrix-matrix products about the possible speedups offered by Numba, and the effects of cache-efficient programming.

## 1.1 Benchmark
Benchmark the above function against the Numpy dot product for matrix sizes up to 1000. Plot the timing results of the above function against the timing results for the Numpy dot product. You need not benchmark every dimension up to 1000. Figure out what dimensions to use so that you can represent the result without spending too much time waiting for the code to finish. To perform benchmarks you can use the `%timeit` magic command. An example is

    ```
    timeit_result = %timeit -o matrix_product(a, b)
    print(timeit_result.best)
    ```

## 1.2 Optimize
Now optimise the code by using Numba to JIT-compile it. Also, there is lots of scope for parallelisation in the code. You can for example parallelize the outer-most for-loop. Benchmark the JIT-compiled serial code against the JIT-compiled parallel code. Comment on the expected performance on your system against the observed performance.

## 1.3 (Optional) Cache Optimization
Now let us improve Cache efficiency. Notice that in the matrix $B$ we traverse by columns. However, the default storage ordering in Numpy is row-based. Hence, the expression `mat_b[k, col_ind]` jumps in memory by `n` units if we move from $k$ to $k+1$. Run your parallelized JIT-compiled Numba code again. But this time choose a matrix $B$ that is stored in column-major order. To change an array to column major order you can use the command `np.asfortranarray`.






In [15]:
from numba import jit, njit, prange

@jit
def matrix_product_compile(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in range(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

@njit(parallel=True)
def matrix_product_parallel(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in prange(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

In [ ]:
dimensions = [2**i for i in range(1, 11)]

python_times = []
numpy_times = []
compile_times = []
parallel_times = []
for d in dimensions:
    print(f"{d} x {d}")
    a = np.random.randn(d, d)
    b = np.random.randn(d, d)

    # Python
    res_py = %timeit -o -q -r 3 matrix_product(a, b)
    python_times.append(res_py.average)

    # Numpy
    res_np = %timeit -o -q -r 3 a @ b
    numpy_times.append(res_np.average)

    # Compiled
    res_comp = %timeit -o -q -r 3 matrix_product_compile(a, b)
    compile_times.append(res_comp.average)

    # Parallel
    res_par = %timeit -o -q -r 3 matrix_product_parallel(a, b)
    parallel_times.append(res_par.average)

print("Benchmarking complete.")
print(f"Python times: {python_times}")
print(f"NumPy times: {numpy_times}")
print(f"Compile times: {compile_times}")
print(f"Parallel times: {parallel_times}")

2 x 2
4 x 4
8 x 8
16 x 16
32 x 32
64 x 64
128 x 128
256 x 256
512 x 512
1024 x 1024


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(dimensions, python_times, marker='o', linestyle='-', label='Custom Python')
plt.plot(dimensions, numpy_times, marker='s', linestyle='--', label='NumPy @')
plt.plot(dimensions, compile_times, marker='^', linestyle='-.', label='Numba JIT-compiled')
plt.plot(dimensions, parallel_times, marker='x', linestyle=':', label='Numba JIT-compiled (parallel)')

#plt.yscale('log')

plt.xlabel('Matrix Dimension (n x n)')
plt.ylabel('Execution Time (seconds)')
plt.title('Matrix Multiplication Performance: Python vs NumPy')

plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)

plt.show()